## N01 – Estacionariedad y Transformaciones


### Celda 0 – Setup general
Inicialización de librerías, directorios y estructura para registrar transformaciones.

In [1]:

import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from pathlib import Path

RESULTS_DIR = Path("results_n01")
RESULTS_DIR.mkdir(exist_ok=True)

TRANSFORM_LOG = []


### Celda 1 – Carga de datos
Carga del dataset diario proveniente del notebook N00.

In [2]:

df = pd.read_excel("results_n00/df_daily.xlsx")
df = df.sort_values("Fecha").reset_index(drop=True)
df.head()


,Fecha,compras_ANCAP,int_ANCAP,var_stock_pet,inv_ANCAP,venta_autos,saldo_cuenta_corriente,ingreso_primario,inversion_directa,inversion_de_cartera,...,dolar_japon,dolar_euro,dolar_libra,dolar_suiza,dolar_canada,dolar_australia,dolar_nueva_zelanda,dolar_sudafrica,DXY_index,usd_dlog
0,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,120.43,0.84,0.65,1.01,1.18,1.24,1.30,11.73,91.38,0.006231
1,2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,118.67,0.84,0.66,1.01,1.18,1.24,1.29,11.72,91.89,0.003307
2,2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,119.29,0.84,0.66,1.01,1.18,1.24,1.29,11.69,92.37,0.005351
3,2015-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,119.79,0.85,0.66,1.02,1.18,1.23,1.28,11.58,91.94,0.004097
4,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,118.21,0.84,0.66,1.01,1.18,1.22,1.27,11.47,91.98,0.004894


### Celda 2 – Identificación de variables numéricas

In [3]:

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols[:10]


['compras_ANCAP',
 'int_ANCAP',
 'var_stock_pet',
 'inv_ANCAP',
 'venta_autos',
 'saldo_cuenta_corriente',
 'ingreso_primario',
 'inversion_directa',
 'inversion_de_cartera',
 'derivados_financieros']

### Celda 3 – Función auxiliar ADF

In [4]:

def adf_pvalue(series):
    try:
        return adfuller(series.dropna(), autolag="AIC")[1]
    except:
        return np.nan


### Celda 4 – Test de estacionariedad en nivel

In [5]:

adf_level = {c: adf_pvalue(df[c]) for c in num_cols}
pd.Series(adf_level).head()


compras_ANCAP    1.371083e-01
int_ANCAP        1.544709e-05
var_stock_pet    2.975804e-10
inv_ANCAP        3.806271e-10
venta_autos      8.092588e-03
dtype: float64

### Celda 5 – Diferenciación si no es estacionaria

In [6]:

df_diff = df.copy()

for c in num_cols:
    if adf_level[c] > 0.05:
        df_diff[c] = df[c].diff()
        TRANSFORM_LOG.append({"variable": c, "transform": "diff_1"})
    else:
        TRANSFORM_LOG.append({"variable": c, "transform": "level"})

df_diff.head()


,Fecha,compras_ANCAP,int_ANCAP,var_stock_pet,inv_ANCAP,venta_autos,saldo_cuenta_corriente,ingreso_primario,inversion_directa,inversion_de_cartera,...,dolar_japon,dolar_euro,dolar_libra,dolar_suiza,dolar_canada,dolar_australia,dolar_nueva_zelanda,dolar_sudafrica,DXY_index,usd_dlog
0,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.84,NaN,NaN,1.18,NaN,NaN,NaN,NaN,0.006231
1,2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.76,0.84,0.01,0.00,1.18,0.00,-0.01,-0.01,0.51,0.003307
2,2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.62,0.84,0.00,0.00,1.18,0.00,0.00,-0.03,0.48,0.005351
3,2015-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.50,0.85,0.00,0.01,1.18,-0.01,-0.01,-0.11,-0.43,0.004097
4,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.58,0.84,0.00,-0.01,1.18,-0.01,-0.01,-0.11,0.04,0.004894


### Celda 6 – Test ADF post diferenciación

In [7]:

adf_diff_res = {c: adf_pvalue(df_diff[c]) for c in num_cols}
pd.Series(adf_diff_res).head()


compras_ANCAP    0.000000e+00
int_ANCAP        1.544709e-05
var_stock_pet    2.975804e-10
inv_ANCAP        3.806271e-10
venta_autos      8.092588e-03
dtype: float64

### Celda 7 – Log-diferencias si sigue no estacionaria

In [8]:

df_final = df_diff.copy()

for c in num_cols:
    if adf_diff_res[c] > 0.05 and (df[c] > 0).all():
        df_final[c] = np.log(df[c]).diff()
        for t in TRANSFORM_LOG:
            if t["variable"] == c:
                t["transform"] = "log_diff"

df_final.head()


,Fecha,compras_ANCAP,int_ANCAP,var_stock_pet,inv_ANCAP,venta_autos,saldo_cuenta_corriente,ingreso_primario,inversion_directa,inversion_de_cartera,...,dolar_japon,dolar_euro,dolar_libra,dolar_suiza,dolar_canada,dolar_australia,dolar_nueva_zelanda,dolar_sudafrica,DXY_index,usd_dlog
0,2015-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.84,NaN,NaN,1.18,NaN,NaN,NaN,NaN,0.006231
1,2015-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.76,0.84,0.01,0.00,1.18,0.00,-0.01,-0.01,0.51,0.003307
2,2015-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.62,0.84,0.00,0.00,1.18,0.00,0.00,-0.03,0.48,0.005351
3,2015-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.50,0.85,0.00,0.01,1.18,-0.01,-0.01,-0.11,-0.43,0.004097
4,2015-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.58,0.84,0.00,-0.01,1.18,-0.01,-0.01,-0.11,0.04,0.004894


### Celda 8 – Limpieza final

In [9]:

df_final = df_final.dropna().reset_index(drop=True)
df_final.head()


,Fecha,compras_ANCAP,int_ANCAP,var_stock_pet,inv_ANCAP,venta_autos,saldo_cuenta_corriente,ingreso_primario,inversion_directa,inversion_de_cartera,...,dolar_japon,dolar_euro,dolar_libra,dolar_suiza,dolar_canada,dolar_australia,dolar_nueva_zelanda,dolar_sudafrica,DXY_index,usd_dlog
0,2015-04-01,0.0,32.41,2039.22,2082.66,4955.0,0.0,0.0,0.0,-1065.9,...,-0.45,0.93,0.00,0.00,1.26,0.00,0.00,-0.14,-0.17,-0.000778
1,2015-04-06,0.0,32.41,2039.22,2082.66,4955.0,0.0,0.0,0.0,-1065.9,...,-0.09,0.91,0.00,-0.01,1.25,0.01,-0.01,-0.19,-1.08,-0.002727
2,2015-04-07,0.0,32.41,2039.22,2082.66,4955.0,0.0,0.0,0.0,-1065.9,...,0.81,0.92,0.00,0.01,1.25,-0.01,0.00,0.08,0.72,0.005059
3,2015-04-08,0.0,32.41,2039.22,2082.66,4955.0,0.0,0.0,0.0,-1065.9,...,-0.20,0.93,0.00,0.00,1.25,-0.01,-0.01,-0.07,0.24,0.008889
4,2015-04-09,0.0,32.41,2039.22,2082.66,4955.0,0.0,0.0,0.0,-1065.9,...,0.46,0.94,0.01,0.01,1.26,0.00,0.00,0.13,1.09,0.006902


### Celda 9 – Export del dataset estacionarizado

In [10]:

df_final.to_excel(RESULTS_DIR / "df_st_daily.xlsx", index=False)


### Celda 10 – Export del tracking de transformaciones

In [11]:

transform_df = pd.DataFrame(TRANSFORM_LOG)
transform_df.to_excel(RESULTS_DIR / "transformaciones_variables.xlsx", index=False)
transform_df.head()


,variable,transform
0,compras_ANCAP,diff_1
1,int_ANCAP,level
2,var_stock_pet,level
3,inv_ANCAP,level
4,venta_autos,level
